# Making Text: Computational Language Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/05-making-text/Making-Text.ipynb)

For four sessions we've been taking text apart: scanning it, sourcing it, splitting it into tokens, turning it into counts. This session runs the pipeline backwards.

We'll build the simplest machine that can write—one that does nothing but count which word followed which—and then turn the same two knobs on a real language model. The question underneath all of it: when a machine produces fluent text, what is it actually doing, and how much of what feels like thinking is just the scale of the counting?

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it. You don't need a GPU; Colab's default CPU runtime is fine. The language model near the end downloads about 500 MB the first time you run it. If your session restarts, choose **Runtime > Run before** on the cell you were working on to catch up.

In [ ]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of our books, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('Making-Text.ipynb'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/05-making-text

## Packages
We'll use a few Python packages to count words and to run a language model:
- [**nltk**](https://www.nltk.org/): The Natural Language Toolkit, which holds the collection of documents we'll compare against our books
- [**transformers**](https://huggingface.co/docs/transformers): Hugging Face's library for running language models, which we'll use for GPT-2
- [**pandas**](https://pandas.pydata.org/): An open source data manipulation and analysis package
- [**matplotlib**](https://matplotlib.org/): A package for drawing charts

In [ ]:
# Packages to install if necessary (they all come preinstalled in Colab)
%pip install -q nltk transformers pandas matplotlib

import gzip # lets us read our compressed copies of the books
import random # lets us sample a word from a distribution
import re # regular expression library, for finding words in text
from collections import Counter, defaultdict # counts things, and makes a dictionary of counters

import matplotlib.pyplot as plt
import nltk
import pandas as pd

BOOK_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"] # blue, orange, aqua: the same colors as our last two workshops
random.seed(0) # so everyone in the room gets the same output the first time through

## The Room as a Model

Before any code: we just did this on the whiteboard. A few words of context went up, the room shouted out what came next, and the tally of those shouts became a distribution over possible next words.

Four things we did by hand have names:

| What we did | What it's called |
|---|---|
| Took the word with the most shouts | **greedy decoding** |
| Rolled a weighted die over the tally | **sampling** |
| Crossed off all but the top three, then rolled | **top-k** |
| Shouted only safe guesses, or anything plausible | **temperature** |

Everything below is that, done faster and on a whole book.

## Counting What Comes Next

An **n-gram** is a run of *n* words. An **n-gram model** is a table: for every context of *n*−1 words, a tally of the words that followed it. There's no training loop and no math beyond counting.

In [ ]:
# Two functions from our previous workshops, unchanged
def find_words(text):
    """Return a list of the lowercase words in a text, keeping contractions like "don't"."""
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower().replace("’", "'"))

def load_book(book_id):
    """Return the text of a saved book, without the Project Gutenberg header and license."""
    with gzip.open(f"../03-Tokenization/texts/{book_id}.txt.gz", "rt", encoding="utf-8") as f:
        text = f.read()
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text).end()
    end = re.search(r"\*\*\* END OF", text).start()
    return text[start:end]

BOOKS = {
    98: "A Tale of Two Cities",
    2701: "Moby-Dick",
    1342: "Pride and Prejudice",
    1260: "Jane Eyre",
    345: "Dracula",
    84: "Frankenstein",
    11: "Alice in Wonderland",
    1661: "Sherlock Holmes",
}

dickens = find_words(load_book(98))
print(f"A Tale of Two Cities: {len(dickens):,} words")

Here is the whole model. Five lines.

In [ ]:
# We'll create a function that counts what follows every context of n-1 words
def build_model(tokens, n):
    """Return a dictionary: each context of n-1 words maps to a Counter of the words that followed it."""
    model = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        context = tuple(tokens[i:i + n - 1]) # the n-1 words of context
        next_word = tokens[i + n - 1] # the word that came after them
        model[context][next_word] += 1 # add one to that word's tally
    return model

trigrams = build_model(dickens, 3)
print(f"{len(trigrams):,} contexts")

✏️ **Try it:** Look up a context and read its tally. This is the model's entire knowledge of what can follow these two words.

In [ ]:
context = ("it", "was") # ✏️ try ("mr", "lorry"), ("the", "guillotine"), or two words of your own
trigrams[context].most_common(8)

Now we need to turn a tally into a word, and a word into a sentence.

In [ ]:
# We'll create a function that takes the last n-1 words we have
def last_context(words, n):
    """Return the final n-1 words as a tuple, or an empty tuple when n is 1."""
    return tuple(words[-(n - 1):]) if n > 1 else ()

# We'll create a function that picks one word out of a Counter
def choose_next(counter, temperature=1.0, top_k=None):
    """Pick a word from a tally of continuations. temperature=0 always takes the most frequent."""
    words, counts = zip(*counter.most_common())
    if top_k: # keep only the top_k most frequent continuations
        words, counts = words[:top_k], counts[:top_k]
    if temperature == 0: # greedy: no randomness at all
        return words[0]
    weights = [count ** (1 / temperature) for count in counts] # reshape the tally
    return random.choices(words, weights=weights, k=1)[0]

# We'll create a function that generates text one word at a time
def generate(model, n, start, length=40, temperature=1.0, top_k=None):
    """Continue the start phrase, choosing one word at a time from the model."""
    words = find_words(start)
    for _ in range(length):
        context = last_context(words, n)
        if context not in model: # we have never seen these words together
            words.append("[dead end]")
            break
        words.append(choose_next(model[context], temperature, top_k))
    return " ".join(words)

We'll seed every model with the sentence this series keeps coming back to. It needs to be at least *n*−1 words long, which is why it's the whole opening clause rather than three words.

✏️ **Predict first:** write down what you expect to change as *n* goes up. Does the text get better? Worse? Both?

In [ ]:
my_prediction = "" # ✏️ type your prediction here before running the next cell

SEED = "it was the best of times it was the worst of times"
for n in (1, 2, 3, 5, 8):
    random.seed(3) # same starting point each time, so only n is changing
    print(f"--- n = {n}")
    print(generate(build_model(dickens, n), n, SEED, length=30), "\n")

**Discuss:** Where did it stop being word salad? Where did it stop being writing? Those are two different values of *n*, and the gap between them is narrow.

In [ ]:
# We'll create a function that asks how many contexts have only one possible continuation
def seen_once(tokens, n):
    """Return the share of contexts that were followed by exactly one word in the whole book."""
    model = build_model(tokens, n)
    only_one = sum(1 for counter in model.values() if sum(counter.values()) == 1)
    return only_one / len(model)

pd.DataFrame(
    {"contexts with one option": [f"{seen_once(dickens, n):.0%}" for n in (1, 2, 3, 5, 8)]},
    index=pd.Index([1, 2, 3, 5, 8], name="n"),
)

**Discuss:** At *n* = 8, every context in the book was followed by exactly one word, so the model has no choices left to make. It isn't writing anymore—it's reciting.

Nothing about this can be fixed with a bigger book. Keep the question for later: *when a model reproduces its training data, what happened?*

## Choosing What Comes Next

A tally is not a decision. The model hands us a distribution; something else has to pick. That something is a **sampling rule**, and it is not part of the model at all—it's a choice we make when we run it.

Start with the simplest rule: always take the most frequent word.

In [ ]:
for n in (2, 3):
    print(f"--- n = {n}, always taking the most frequent word")
    print(generate(build_model(dickens, n), n, SEED, length=80, temperature=0), "\n")

**Discuss:** Why does it do that? What would you have to change to stop it?

**Temperature** reshapes the tally before we sample. We raise every count to the power 1/*T*: below 1 the favorites pull further ahead, above 1 everything flattens toward equally likely.

In [ ]:
for temperature in (0.3, 1.0, 2.0):
    random.seed(5)
    print(f"--- temperature = {temperature}")
    print(generate(trigrams, 3, SEED, length=30, temperature=temperature), "\n")

In [ ]:
# We'll create a function that draws one context's tally at three temperatures
def plot_temperatures(counter, temperatures=(0.3, 1.0, 2.0), top=6):
    """Draw the reweighted distribution over the most common continuations."""
    words, counts = zip(*counter.most_common(top))
    fig, axes = plt.subplots(1, len(temperatures), figsize=(11, 3), sharey=True)
    for ax, temperature, color in zip(axes, temperatures, BOOK_COLORS):
        weights = [count ** (1 / temperature) for count in counts]
        shares = [weight / sum(weights) for weight in weights]
        ax.bar(words, shares, color=color, edgecolor="white", linewidth=1)
        ax.set_title(f"temperature = {temperature}", loc="left")
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(axis="x", labelrotation=45)
    axes[0].set_ylabel("chance of being chosen")
    plt.tight_layout()
    plt.show()

plot_temperatures(trigrams[("it", "was")])

**top-k** is blunter: throw away everything but the *k* most frequent continuations, then sample from what's left.

In [ ]:
for top_k in (1, 3, 25):
    random.seed(5)
    print(f"--- top_k = {top_k}")
    print(generate(trigrams, 3, SEED, length=30, top_k=top_k), "\n")

**Discuss:** Usable text sits between two failure modes—repetition on one side, noise on the other. Where is the edge for this model, and would you put it somewhere else for a different purpose?

One thing worth being precise about: turning these knobs changes *nothing* about the model. The tallies are identical. We are only changing how we read them.

## Two Collections

The model knows one collection. Swap it and you swap everything the model can say, so the collection isn't an input to the model—it *is* the model.

The Brown Corpus holds 500 documents in fifteen categories. We used it last session to test whether counting could sort documents by genre. The categories are very different sizes.

In [ ]:
nltk.download("brown", quiet=True)
from nltk.corpus import brown

sizes = {c: len([w for w in brown.words(categories=c) if w.isalpha()]) for c in brown.categories()}
pd.Series(sizes, name="words").sort_values(ascending=False).to_frame()

✏️ **Predict first:** which category will produce the most fluent-sounding text? Write down your guess and your reason before running the next two cells.

In [ ]:
my_prediction = "" # ✏️ which category, and why?

In [ ]:
# We'll create a function that builds a model from one Brown category
def brown_model(category, n=3):
    """Return an n-gram model built from every document in one Brown category."""
    words = [word.lower() for word in brown.words(categories=category) if word.isalpha()]
    return build_model(words, n)

for category in ("learned", "science_fiction"): # the largest and the smallest
    random.seed(11)
    print(f"--- {category} ({sizes[category]:,} words)")
    print(generate(brown_model(category), 3, "it is not", length=30), "\n")

**Discuss:** Most people predict the largest category. Read those two again—which actually sounds more like a person wrote it? And if it's the one with thirteen times less text, what is the model doing?

In [ ]:
# We'll create a function that finds the longest run of generated words copied straight from the source
def longest_copied(generated, source_words):
    """Return the longest stretch of the generated text that appears verbatim in the source."""
    source = " " + " ".join(source_words) + " "
    best, best_run = 0, ""
    words = generated.split()
    for i in range(len(words)):
        for j in range(i + best + 1, len(words) + 1):
            run = " ".join(words[i:j])
            if " " + run + " " in source: # this stretch came straight from the book
                best, best_run = j - i, run
            else:
                break
    return best, best_run

for category in ("learned", "government", "science_fiction"):
    words = [w.lower() for w in brown.words(categories=category) if w.isalpha()]
    random.seed(0)
    text = generate(build_model(words, 3), 3, "it is not", length=40)
    length, run = longest_copied(text, words)
    print(f"{category:>16} ({sizes[category]:>7,} words): copied {length} words in a row")
    print(f"{'':>18} \"{run}\"\n")

**Discuss:** The less text a model has, the longer the stretches it copies. And copied text reads well, because a person wrote it.

This is the *n* = 8 result arriving from a second direction. Fluency is not evidence that a model understands anything—sometimes it's evidence that it's quoting.

Size isn't the only thing a collection decides. Matched for size, these two are almost equally predictable, and they still sound nothing alike.

In [ ]:
learned_words = [w.lower() for w in brown.words(categories="learned") if w.isalpha()]
matched = min(len(dickens), len(learned_words))

for name, words in (("A Tale of Two Cities", dickens[:matched]), ("Brown learned", learned_words[:matched])):
    random.seed(4)
    print(f"--- {name} ({matched:,} words, {seen_once(words, 3):.0%} of contexts have one option)")
    print(generate(build_model(words, 3), 3, "it is not", length=28), "\n")

**Discuss:** One is a single author with a consistent voice; the other is dozens of authors on unrelated subjects. The numbers can't tell them apart. Your ear can, instantly. What is your ear using that the counting isn't?

## What Counting Can't Do

Our model can only see *n*−1 words back. Anything that depends on information further away than that is invisible to it, no matter how much text we add.

In [ ]:
LONG_RANGE = "the prisoner who had been held in the bastille for eighteen years was"
random.seed(2)
print(generate(build_model(dickens, 3), 3, LONG_RANGE, length=25))

Two things change in a transformer-based model, and only two are worth naming here:

1. **The window isn't fixed.** Any position can draw on any earlier position, so the dependency range is the whole context rather than *n*−1 words.
2. **Representations are learned, not counted.** To our model, *stone* and *rock* are unrelated dictionary keys—exactly the problem we hit at the end of last session. In a transformer each word is a vector in a learned space, so what it knows about one word partly transfers to another.

Everything else about the interface is the same: a distribution over next words, and a sampling rule to pick from it.

## The Same Knobs, A Bigger Machine

GPT-2 was released in 2019 with 124 million parameters, trained on about 40 GB of web pages. It is small and old enough to run on a laptop, and it is a real transformer.

The arguments below are the same two we just invented.

In [ ]:
from transformers import pipeline

writer = pipeline("text-generation", model="gpt2") # downloads about 500 MB the first time

In [ ]:
#@title Turn the knobs, then run { run: "auto" }
prompt = "It was the best of times, it was the worst of times" #@param {type:"string"}
temperature = 0.8 #@param {type:"slider", min:0.1, max:2.0, step:0.1}
top_k = 40 #@param {type:"slider", min:1, max:100, step:1}

output = writer(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=temperature,
    top_k=int(top_k),
)
print(output[0]["generated_text"])

✏️ **Try it:** give GPT-2 the long-range sentence our n-gram couldn't finish.

In [ ]:
print(writer(LONG_RANGE, max_new_tokens=40, do_sample=True, temperature=0.8, top_k=40)[0]["generated_text"])

**Discuss:** Four machines, one operation.

| | trained on | contexts |
|---|---|---|
| The room, on the whiteboard | whatever we've all read | one |
| Our n-gram | one novel | ~10<sup>5</sup> |
| GPT-2 | ~40 GB of web pages | 124 million parameters |
| A current assistant | far more | far more |

Every rung produces a distribution over next words and needs a rule to pick from it. That part really is the same all the way up. What changed is the window, the representations, and the sheer amount of text the distribution absorbed—enough that its guesses start to look like thought.

## Your Turn

**Before you start:** run the catch-up cell below. If you've run every cell above, it changes nothing.

In [ ]:
# Catch-up cell: restores everything the activities need if your session restarted
dickens = find_words(load_book(98))
trigrams = build_model(dickens, 3)
SEED = "it was the best of times it was the worst of times"
LONG_RANGE = "the prisoner who had been held in the bastille for eighteen years was"
nltk.download("brown", quiet=True)
from nltk.corpus import brown
print("ready")

### Activity 1: Pick a Book

Build a model from a different book and find the *n* where it stops writing and starts reciting. It won't be the same number as Dickens.

In [ ]:
my_book = 2701 # ✏️ try 1342, 1260, 345, 84, 11, or 1661
my_tokens = find_words(load_book(my_book))
my_seed = "it was a" # ✏️ needs to be at least n-1 words long

for n in (2, 3, 5, 8):
    random.seed(3)
    print(f"--- n = {n}  ({seen_once(my_tokens, n):.0%} of contexts have one option)")
    print(generate(build_model(my_tokens, n), n, my_seed, length=25), "\n")

### Activity 2: Find the Edge

Hold *n* at 3 and find the temperature where the text stops being readable, and the one where it starts repeating itself.

In [ ]:
my_temperature = 1.0 # ✏️ try values between 0.1 and 2.5
random.seed(5)
print(generate(trigrams, 3, SEED, length=40, temperature=my_temperature))

### Activity 3: Two Collections

Train on two Brown categories and compare. Which pairs are easiest to tell apart from the output alone?

In [ ]:
for category in ("learned", "humor"): # ✏️ try "editorial", "hobbies", "religion", "fiction", "lore"
    random.seed(11)
    print(f"--- {category}")
    print(generate(brown_model(category), 3, "it is not", length=30), "\n")

### Activity 4: Ask the Big Model

Give GPT-2 something your n-gram would have no chance with: a question, an instruction, the start of an argument.

In [ ]:
my_prompt = "The three most important things about libraries are" # ✏️ replace this
print(writer(my_prompt, max_new_tokens=60, do_sample=True, temperature=0.8, top_k=40)[0]["generated_text"])

## Wrapping Up: What Did the Machine Add?

Every session in this series has been about subtraction. Scanning lost the page. OCR lost characters. Tokenizing lost punctuation and case. Counting lost word order. Each step traded something away to make the text computable, and we kept a running tally of what went missing.

This session inverts it. Nothing was lost here. Text was *added*—words that appear in no source, assembled out of the statistical residue of everything that came before.

The losses from the first four sessions are still missing. The machine fills those gaps with invention and hands it back in fluent prose, and it has no way to mark which is which. That is the whole problem, and you just built the smallest working version of it.